# ChromHMM Parameters Analysis — ENCODE (Chromosome 1)

This notebook evaluates the variability of ChromHMM chromatin state segmentations across parameter configurations. All models have 15 states, matching the reference segmentation:
1. **Intra-Sample (Method) Variance vs. Cross-Sample (Biological) Variance**: Pairwise Adjusted Mutual Information (AMI) comparing segmentations of the same sample under diverse parameter variations against distinct biological samples under identical parameter configurations.
2. **Reference Matching**: Optimal state matching of each parameter configuration to the corresponding published GSE/ENCODE reference segmentation on chromosome 1.
3. **Non-quiescent Share**: Quantification of active chromatin genome coverage (annotated non-quiescent fraction) across samples and parameter configurations.
4. **Reference Agreement (Composition, Kappa, Jaccard, AMI)**: Multi-metric concordance against the reference over all states. Composition is the cosine of the bp-per-state vectors over every state except the biggest one of either side, so the dominant quiescent state does not saturate it.
5. **Intra-Sample & Cross-Sample Agreements (Composition, Kappa, Jaccard, AMI)**: Comprehensive multi-metric concordance distributions for intra-sample method variations and cross-sample biological comparisons.

All computations, binarizations, models, and exported data are stored under `~/data/2026_segmentations/encode/variance`.

In [1]:
# Setup, imports, and environment configuration
import glob
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

# Locate project root and load configuration
PROJECT_ROOT = None
for _p in [os.getcwd()] + sys.path:
    _curr = os.path.abspath(_p) if _p else ""
    while _curr and _curr != os.path.dirname(_curr):
        if os.path.exists(os.path.join(_curr, "scripts/analysis/analyze.py")):
            PROJECT_ROOT = _curr
            break
        _curr = os.path.dirname(_curr)
    if PROJECT_ROOT:
        break
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()

config_path = os.path.join(PROJECT_ROOT, "config_encode.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

for sub in ["scripts/analysis", "scripts/rules"]:
    s_dir = os.path.abspath(os.path.join(PROJECT_ROOT, sub))
    if s_dir not in sys.path:
        sys.path.insert(0, s_dir)

import match
import summary_plots
import utils
utils.set_style()

workdir = os.path.expanduser(config.get("workdir", "~/data/2026_segmentations/encode"))
parameters_dir = os.path.join(workdir, "parameters")
plots_dir = os.path.join(parameters_dir, "plots")
os.makedirs(plots_dir, exist_ok=True)

chromsizes_chr1 = os.path.join(parameters_dir, "chr1.chrom.sizes")
with open(chromsizes_chr1, "w") as f:
    f.write("chr1\t248956422\n")

chromhmm_jar = os.path.join(workdir, "ChromHMM/ChromHMM.jar")
if not os.path.exists(chromhmm_jar):
    chromhmm_jar = os.path.expanduser("~/data/2026_segmentations/ChromHMM/ChromHMM.jar")

print(f"Project root : {PROJECT_ROOT}")
print(f"Working dir  : {workdir}")
print(f"Parameters dir : {parameters_dir}")
print(f"ChromHMM JAR : {chromhmm_jar}")

Project root : /Users/Oleg.Shpynov/work/omni-chromhmm
Working dir  : /Users/Oleg.Shpynov/data/2026_segmentations/encode
Parameters dir : /Users/Oleg.Shpynov/data/2026_segmentations/encode/parameters
ChromHMM JAR : /Users/Oleg.Shpynov/data/2026_segmentations/encode/ChromHMM/ChromHMM.jar


In [2]:
# Define datasets, marks, and parameter variation matrix
SAMPLES = ["imr90", "monocytes", "adrenal_gland"]
CELL_NAMES = {
    "imr90": "IMR90",
    "monocytes": "Monocytes",
    "adrenal_gland": "AdrenalGland",
}
MARKS = ["H3K4me3", "H3K27ac", "H3K4me1", "H3K36me3", "H3K9me3", "H3K27me3"]

# Binarization parameter configurations
BIN_CONFIGS = [
    ("ctrl_p1e4", True, ["-p", "0.0001", "-b", "200"], "Default (Ctrl, p=1e-4)"),
    ("no_ctrl_p1e4", False, ["-p", "0.0001", "-b", "200"], "Default (No-Ctrl, p=1e-4)"),
    ("ctrl_p1e3", True, ["-p", "0.001", "-b", "200"], "Ctrl, p=1e-3"),
    ("ctrl_p1e5", True, ["-p", "0.00001", "-b", "200"], "Ctrl, p=1e-5"),
    ("no_ctrl_p1e3", False, ["-p", "0.001", "-b", "200"], "No-Ctrl, p=1e-3"),
    ("no_ctrl_p1e5", False, ["-p", "0.00001", "-b", "200"], "No-Ctrl, p=1e-5"),
    ("ctrl_shift100", True, ["-p", "0.0001", "-n", "100", "-b", "200"], "Ctrl, Shift 100bp"),
    ("ctrl_shift0", True, ["-p", "0.0001", "-n", "0", "-b", "200"], "Ctrl, Shift 0bp"),
]

# LearnModel parameter configurations, all at 15 states so every model has the
# state count of the reference segmentation
LEARN_CONFIGS = [
    ("default_ctrl", "ctrl_p1e4", 15, 123, [], "Default Ctrl (15st, s123)", "Baseline"),
    ("default_no_ctrl", "no_ctrl_p1e4", 15, 123, [], "Default No-Ctrl (15st, s123)", "Control Effect"),
    ("bin_ctrl_p1e3", "ctrl_p1e3", 15, 123, [], "Binarize p=1e-3 (Ctrl)", "Binarization P-value"),
    ("bin_ctrl_p1e5", "ctrl_p1e5", 15, 123, [], "Binarize p=1e-5 (Ctrl)", "Binarization P-value"),
    ("bin_no_ctrl_p1e3", "no_ctrl_p1e3", 15, 123, [], "Binarize p=1e-3 (No-Ctrl)", "Binarization P-value"),
    ("bin_no_ctrl_p1e5", "no_ctrl_p1e5", 15, 123, [], "Binarize p=1e-5 (No-Ctrl)", "Binarization P-value"),
    ("bin_ctrl_shift100", "ctrl_shift100", 15, 123, [], "Read Shift 100bp (Ctrl)", "Read Shift"),
    ("bin_ctrl_shift0", "ctrl_shift0", 15, 123, [], "Read Shift 0bp (Ctrl)", "Read Shift"),
    ("learn_seed456", "ctrl_p1e4", 15, 456, [], "Seed 456 (Ctrl)", "Random Seed"),
    ("learn_seed789", "ctrl_p1e4", 15, 789, [], "Seed 789 (Ctrl)", "Random Seed"),
    ("learn_init_random", "ctrl_p1e4", 15, 123, ["-init", "random"], "Random Init (Ctrl)", "Initialization"),
    ("learn_d1e2", "ctrl_p1e4", 15, 123, ["-d", "0.01"], "Convergence Delta 1e-2", "Convergence"),
    ("learn_maxiter50", "ctrl_p1e4", 15, 123, ["-r", "50"], "Max Iterations 50", "Convergence"),
    ("no_ctrl_seed456", "no_ctrl_p1e4", 15, 456, [], "Seed 456 (No-Ctrl)", "Random Seed (No-Ctrl)"),
    ("no_ctrl_init_random", "no_ctrl_p1e4", 15, 123, ["-init", "random"], "Random Init (No-Ctrl)", "Initialization (No-Ctrl)"),
]

METHOD_NAMES = {m[0]: m[5] for m in LEARN_CONFIGS}
METHOD_CATEGORIES = {m[0]: m[6] for m in LEARN_CONFIGS}

print(f"Configured {len(SAMPLES)} samples:")
for s in SAMPLES:
    print(f"  - {s} ({CELL_NAMES[s]})")
print(f"Configured {len(BIN_CONFIGS)} binarization settings.")
print(f"Configured {len(LEARN_CONFIGS)} LearnModel parameter settings.")

Configured 3 samples:
  - imr90 (IMR90)
  - monocytes (Monocytes)
  - adrenal_gland (AdrenalGland)
Configured 8 binarization settings.
Configured 15 LearnModel parameter settings.


In [3]:
# Create cellmarkfiletable and ensure Chromosome 1 BAMs are prepared
def create_cellmark_tables():
    for ds in SAMPLES:
        cname = CELL_NAMES[ds]
        # With control table
        tbl_ctrl = os.path.join(parameters_dir, f"{ds}_table_ctrl.tsv")
        with open(tbl_ctrl, "w") as f:
            for m in MARKS:
                f.write(f"{cname}\t{m}\t{m}.bam\t{m}.bam\n")
        # Without control table
        tbl_no_ctrl = os.path.join(parameters_dir, f"{ds}_table_no_ctrl.tsv")
        with open(tbl_no_ctrl, "w") as f:
            for m in MARKS:
                f.write(f"{cname}\t{m}\t{m}.bam\n")

create_cellmark_tables()

# Helper to ensure chr1 BAMs exist
def ensure_chr1_bams(ds):
    bams_dir = os.path.join(workdir, ds, "bams_chr1")
    ctrls_dir = os.path.join(workdir, ds, "controls_chr1")
    os.makedirs(bams_dir, exist_ok=True)
    os.makedirs(ctrls_dir, exist_ok=True)

    # Check if files already present
    bams = glob.glob(f"{bams_dir}/*.bam")
    ctrls = glob.glob(f"{ctrls_dir}/*.bam")
    if len(bams) == len(MARKS) and len(ctrls) == len(MARKS):
        return True

    # Extract from full bams if needed
    for sub, out_dir in [("bams", bams_dir), ("controls", ctrls_dir)]:
        src_dir = os.path.join(workdir, ds, sub)
        for m in MARKS:
            src_bam = os.path.join(src_dir, f"{m}.bam")
            dst_bam = os.path.join(out_dir, f"{m}.bam")
            if os.path.exists(src_bam) and (not os.path.exists(dst_bam) or os.path.getsize(dst_bam) == 0):
                if not os.path.exists(src_bam + ".bai"):
                    subprocess.run(["samtools", "index", src_bam], check=True)
                subprocess.run(["samtools", "view", "-b", src_bam, "chr1", "-o", dst_bam], check=True)
                subprocess.run(["samtools", "index", dst_bam], check=True)
    return True

for ds in SAMPLES:
    ensure_chr1_bams(ds)
    n_b = len(glob.glob(os.path.join(workdir, ds, "bams_chr1", "*.bam")))
    n_c = len(glob.glob(os.path.join(workdir, ds, "controls_chr1", "*.bam")))
    print(f"Sample {ds:15s}: {n_b} signal BAMs, {n_c} control BAMs ready on chr1.")

Sample imr90          : 6 signal BAMs, 6 control BAMs ready on chr1.
Sample monocytes      : 6 signal BAMs, 6 control BAMs ready on chr1.
Sample adrenal_gland  : 6 signal BAMs, 6 control BAMs ready on chr1.


In [4]:
# BinarizeBam and LearnModel execution functions with caching
def run_binarize_task(task):
    ds, bname, use_ctrl, extra_args, _ = task
    cname = CELL_NAMES[ds]
    out_dir = os.path.join(parameters_dir, ds, "binarized", bname)
    bin_file = os.path.join(out_dir, f"{cname}_chr1_binary.txt")
    if os.path.exists(bin_file) and os.path.getsize(bin_file) > 1000:
        return f"Cached: {ds} {bname}"
    os.makedirs(out_dir, exist_ok=True)
    tbl = os.path.join(parameters_dir, f"{ds}_table_ctrl.tsv" if use_ctrl else f"{ds}_table_no_ctrl.tsv")
    bamdir = os.path.join(workdir, ds, "bams_chr1")
    ctrldir = os.path.join(workdir, ds, "controls_chr1")

    cmd = ["java", "-mx2000M", "-jar", chromhmm_jar, "BinarizeBam"]
    cmd += extra_args
    if use_ctrl:
        cmd += ["-c", ctrldir]
    cmd += [chromsizes_chr1, bamdir, tbl, out_dir]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return f"Completed: {ds} {bname}"

def run_learn_task(task):
    ds, mname, bname, nstates, seed, extra_args, _, _ = task
    cname = CELL_NAMES[ds]
    in_bin_dir = os.path.join(parameters_dir, ds, "binarized", bname)
    out_model_dir = os.path.join(parameters_dir, ds, "models", mname)
    dense_beds = glob.glob(os.path.join(out_model_dir, "*_dense.bed"))
    if dense_beds and os.path.getsize(dense_beds[0]) > 1000:
        return f"Cached: {ds} {mname}"
    os.makedirs(out_model_dir, exist_ok=True)
    cmd = ["java", "-mx2000M", "-jar", chromhmm_jar, "LearnModel",
           "-p", "1", "-b", "200", "-s", str(seed)]
    cmd += extra_args
    cmd += [in_bin_dir, out_model_dir, str(nstates), "hg38"]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return f"Completed: {ds} {mname}"

# Execute all binarization and model learning tasks
bin_tasks = [(ds, *cfg) for ds in SAMPLES for cfg in BIN_CONFIGS]
print(f"Checking / executing {len(bin_tasks)} BinarizeBam configurations...")
with ThreadPoolExecutor(max_workers=min(8, os.cpu_count())) as ex:
    for res in ex.map(run_binarize_task, bin_tasks):
        pass
print("All binarizations ready.")

learn_tasks = [(ds, *cfg) for ds in SAMPLES for cfg in LEARN_CONFIGS]
print(f"Checking / executing {len(learn_tasks)} LearnModel configurations...")
with ThreadPoolExecutor(max_workers=min(8, os.cpu_count())) as ex:
    for res in ex.map(run_learn_task, learn_tasks):
        pass
print("All learned models ready.")

Checking / executing 24 BinarizeBam configurations...
All binarizations ready.
Checking / executing 45 LearnModel configurations...
All learned models ready.


In [5]:
# Discretize segmentations onto 200bp genomic bins and compute Adjusted Mutual Information (AMI)
BIN_SIZE = 200
CHR1_LEN = 248956422
N_BINS = CHR1_LEN // BIN_SIZE

def load_segmentation_bins(bed_path):
    df = pd.read_csv(bed_path, sep="\t", header=None, skiprows=1, usecols=[0,1,2,3], names=["chr", "start", "end", "state"])
    arr = np.zeros(N_BINS, dtype=np.int32)
    for _, row in df.iterrows():
        s_idx = int(row["start"]) // BIN_SIZE
        e_idx = min(N_BINS, int(row["end"]) // BIN_SIZE)
        state_id = int(str(row["state"]).replace("E", "").split("_")[0])
        arr[s_idx:e_idx] = state_id
    return arr

def pair_ami(bins1, bins2):
    """AMI of two bin vectors, off their overlap table, as match.agreement_metrics() reads it."""
    codes, counts = np.unique(bins1.astype(np.int64) * 1000 + bins2, return_counts=True)
    return match.adjusted_mutual_information({(c // 1000, c % 1000): n for c, n in zip(codes, counts)})

# Load bin vectors for all models
seg_vectors = {ds: {} for ds in SAMPLES}
for ds in SAMPLES:
    for mname, _, _, _, _, _, _ in LEARN_CONFIGS:
        model_dir = os.path.join(parameters_dir, ds, "models", mname)
        dense_beds = glob.glob(os.path.join(model_dir, "*_dense.bed"))
        if not dense_beds:
            print(f"Warning: missing dense BED in {model_dir}")
            continue
        seg_vectors[ds][mname] = load_segmentation_bins(dense_beds[0])

print(f"Successfully loaded {sum(len(v) for v in seg_vectors.values())} segmentations ({N_BINS:,} bins each).")

# 1. Intra-sample pairwise AMI (same sample, different methods)
intra_records = []
mkeys = [m[0] for m in LEARN_CONFIGS]

for ds in SAMPLES:
    for i in range(len(mkeys)):
        for j in range(i + 1, len(mkeys)):
            m1, m2 = mkeys[i], mkeys[j]
            ami = pair_ami(seg_vectors[ds][m1], seg_vectors[ds][m2])
            intra_records.append({
                "Sample": ds,
                "SampleName": CELL_NAMES[ds],
                "Method1": m1,
                "Method2": m2,
                "Method1_Label": METHOD_NAMES[m1],
                "Method2_Label": METHOD_NAMES[m2],
                "Category1": METHOD_CATEGORIES[m1],
                "Category2": METHOD_CATEGORIES[m2],
                "AMI": ami,
                "Comparison": "Intra-Sample (Methods)",
            })

df_intra_ami = pd.DataFrame(intra_records)

# 2. Cross-sample pairwise AMI (different samples, same method)
cross_records = []
sample_pairs = list(combinations(SAMPLES, 2))

for mname, _, _, _, _, label, cat in LEARN_CONFIGS:
    for ds1, ds2 in sample_pairs:
        ami = pair_ami(seg_vectors[ds1][mname], seg_vectors[ds2][mname])
        cross_records.append({
            "Pair": f"{CELL_NAMES[ds1]} vs {CELL_NAMES[ds2]}",
            "Sample1": ds1,
            "Sample2": ds2,
            "Method": mname,
            "Method_Label": label,
            "Category": cat,
            "AMI": ami,
            "Comparison": "Cross-Sample (Biological)",
        })

df_cross_ami = pd.DataFrame(cross_records)

# Save summary tables
df_intra_ami.to_csv(os.path.join(parameters_dir, "intra_sample_ami.tsv"), sep="\t", index=False)
df_cross_ami.to_csv(os.path.join(parameters_dir, "cross_sample_ami.tsv"), sep="\t", index=False)

print("\n--- Summary Statistics ---")
print("Intra-sample AMI (Mean ± SD across all method pairs):")
for ds in SAMPLES:
    sub = df_intra_ami[df_intra_ami["Sample"] == ds]["AMI"]
    print(f"  {CELL_NAMES[ds]:15s}: {sub.mean():.3f} ± {sub.std():.3f} (min={sub.min():.3f}, max={sub.max():.3f})")

print("\nCross-sample AMI (Mean ± SD across same-method pairs):")
for pair in df_cross_ami["Pair"].unique():
    sub = df_cross_ami[df_cross_ami["Pair"] == pair]["AMI"]
    print(f"  {pair:25s}: {sub.mean():.3f} ± {sub.std():.3f} (min={sub.min():.3f}, max={sub.max():.3f})")

Successfully loaded 45 segmentations (1,244,782 bins each).

--- Summary Statistics ---
Intra-sample ARI (Mean ± SD across all method pairs):
  IMR90          : 0.678 ± 0.162 (min=0.383, max=1.000)
  Monocytes      : 0.720 ± 0.126 (min=0.495, max=1.000)
  AdrenalGland   : 0.483 ± 0.230 (min=0.113, max=1.000)

Cross-sample ARI (Mean ± SD across same-method pairs):
  IMR90 vs Monocytes       : 0.185 ± 0.056 (min=0.120, max=0.290)
  IMR90 vs AdrenalGland    : 0.181 ± 0.062 (min=0.086, max=0.317)
  Monocytes vs AdrenalGland: 0.088 ± 0.033 (min=0.057, max=0.160)


In [6]:
# Matching to GSE/ENCODE Reference and Computing Non-quiescent Share
ref_nq_records = []
method_nq_records = []
matched_segmentations = {ds: {} for ds in SAMPLES}

for ds in SAMPLES:
    ref_id = config["datasets"][ds]["ref_chromhmm"]
    ref_bed = os.path.join(workdir, ds, f"{ref_id}_chromhmm.bed")
    ref_segs = [r for r in match.load_bed(ref_bed) if r[0] == "chr1"]
    l_ref = match.state_lengths(ref_segs)
    tot_ref = sum(l_ref.values())
    quies_ref = sum(l_ref.get(s, 0) for s in l_ref if utils.is_quiescent(s))
    ref_nq = (tot_ref - quies_ref) / tot_ref if tot_ref else 0
    
    ref_nq_records.append({
        "Sample": ds,
        "SampleName": CELL_NAMES[ds],
        "Method": "Reference",
        "Method_Label": "Reference (GSE/ENCODE)",
        "Type": "Reference",
        "Total_bp": tot_ref,
        "Quiescent_bp": quies_ref,
        "Quiescent_Share": quies_ref / tot_ref if tot_ref else 0,
        "Non_Quiescent_Share": ref_nq,
    })
    
    for mname, _, _, _, _, label, cat in LEARN_CONFIGS:
        model_dir = os.path.join(parameters_dir, ds, "models", mname)
        dense_beds = glob.glob(os.path.join(model_dir, "*_dense.bed"))
        if not dense_beds:
            continue
        work_segs = match.load_bed(dense_beds[0])
        
        # Hungarian state matching to reference
        mapping = match.match_states(ref_segs, work_segs)
        remapped_work = [(c, s, e, mapping.get(name, name), col) for (c, s, e, name, col) in work_segs]
        matched_segmentations[ds][mname] = (remapped_work, ref_segs)
        
        # Non-quiescent share
        l_work = match.state_lengths(remapped_work)
        tot_work = sum(l_work.values())
        quies_work = sum(l_work.get(s, 0) for s in l_work if utils.is_quiescent(s))
        non_quies_work = (tot_work - quies_work) / tot_work if tot_work else 0
        
        method_nq_records.append({
            "Sample": ds,
            "SampleName": CELL_NAMES[ds],
            "Method": mname,
            "Method_Label": label,
            "Category": cat,
            "Type": "Processed",
            "Total_bp": tot_work,
            "Quiescent_bp": quies_work,
            "Quiescent_Share": quies_work / tot_work if tot_work else 0,
            "Non_Quiescent_Share": non_quies_work,
        })

df_ref_nq = pd.DataFrame(ref_nq_records)
df_method_nq = pd.DataFrame(method_nq_records)
df_all_nq = pd.concat([df_ref_nq, df_method_nq], ignore_index=True)

# Save summary table
nq_tsv = os.path.join(parameters_dir, "non_quiescent_share.tsv")
df_all_nq.to_csv(nq_tsv, sep="\t", index=False)
print(f"Non-quiescent share table saved to: {nq_tsv}")

print("\n--- Non-quiescent Share Summary ---")
for ds in SAMPLES:
    r_val = df_ref_nq[df_ref_nq["Sample"] == ds]["Non_Quiescent_Share"].values[0]
    m_vals = df_method_nq[df_method_nq["Sample"] == ds]["Non_Quiescent_Share"]
    print(f"  {CELL_NAMES[ds]:15s}: Reference = {r_val:.3f} | Processed Mean ± SD = {m_vals.mean():.3f} ± {m_vals.std():.3f} (min={m_vals.min():.3f}, max={m_vals.max():.3f})")

avg_similarity = 0.5025 (jaccard) 0.5025 (quality) over 15/15 matched states, bp accuracy 0.6697
  1 -> Biv  (jaccard=0.5709, quality=0.5709)
  2 -> Enh1  (jaccard=0.0001, quality=0.0001)
  3 -> ReprPC  (jaccard=0.4291, quality=0.4291)
  4 -> Quies  (jaccard=0.6655, quality=0.6655)
  5 -> Het  (jaccard=0.7497, quality=0.7497)
  6 -> ZNF/Rpts  (jaccard=0.4736, quality=0.4736)
  7 -> Tx  (jaccard=0.6666, quality=0.6666)
  8 -> EnhG2  (jaccard=0.0083, quality=0.0083)
  9 -> EnhG1  (jaccard=0.3216, quality=0.3216)
  10 -> TxWk  (jaccard=0.5408, quality=0.5408)
  11 -> Enh2  (jaccard=0.6725, quality=0.6725)
  12 -> TssFlnkU  (jaccard=0.5591, quality=0.5591)
  13 -> TssFlnkD  (jaccard=0.3938, quality=0.3938)
  14 -> TssFlnk  (jaccard=0.6190, quality=0.6190)
  15 -> Tss  (jaccard=0.8662, quality=0.8662)
avg_similarity = 0.4413 (jaccard) 0.4413 (quality) over 15/15 matched states, bp accuracy 0.7086
  1 -> Tx  (jaccard=0.5645, quality=0.5645)
  2 -> ZNF/Rpts  (jaccard=0.0007, quality=0.0007)
 

Non-quiescent share table saved to: /Users/Oleg.Shpynov/data/2026_segmentations/encode/parameters/non_quiescent_share.tsv

--- Non-quiescent Share Summary ---
  IMR90          : Reference = 0.386 | Processed Mean ± SD = 0.533 ± 0.055 (min=0.429, max=0.672)
  Monocytes      : Reference = 0.249 | Processed Mean ± SD = 0.348 ± 0.063 (min=0.250, max=0.456)
  AdrenalGland   : Reference = 0.038 | Processed Mean ± SD = 0.322 ± 0.126 (min=0.080, max=0.565)


avg_similarity = 0.2581 (jaccard) 0.2581 (quality) over 15/15 matched states, bp accuracy 0.9375
  1 -> TxWk  (jaccard=0.2660, quality=0.2660)
  2 -> Biv  (jaccard=0.0370, quality=0.0370)
  3 -> ZNF/Rpts  (jaccard=0.4466, quality=0.4466)
  4 -> Het  (jaccard=0.1981, quality=0.1981)
  5 -> Quies  (jaccard=0.9527, quality=0.9527)
  6 -> Tx  (jaccard=0.1248, quality=0.1248)
  7 -> EnhG1  (jaccard=0.0284, quality=0.0284)
  8 -> Enh2  (jaccard=0.0000, quality=0.0000)
  9 -> Enh1  (jaccard=0.3693, quality=0.3693)
  10 -> TssFlnkU  (jaccard=0.0000, quality=0.0000)
  11 -> Tss  (jaccard=0.8886, quality=0.8886)
  12 -> TssFlnk  (jaccard=0.4720, quality=0.4720)
  13 -> TssFlnkD  (jaccard=0.0663, quality=0.0663)
  14 -> EnhG2  (jaccard=0.0000, quality=0.0000)
  15 -> ReprPC  (jaccard=0.0211, quality=0.0211)
avg_similarity = 0.1422 (jaccard) 0.1422 (quality) over 15/15 matched states, bp accuracy 0.5263
  1 -> TssFlnk  (jaccard=0.2697, quality=0.2697)
  2 -> TssFlnkD  (jaccard=0.0334, quality=0.03

In [ ]:
# Compute Reference Agreement (Composition, Kappa, Jaccard, AMI)
# Composition is the cosine of the bp-per-state vectors without the biggest state
AGREEMENT_METRICS = [
    (utils.COSINE, utils.COMPOSITION_DISPLAY),
    (utils.KAPPA, utils.KAPPA_DISPLAY),
    (utils.JACCARD, utils.JACCARD_DISPLAY),
    (utils.AMI, utils.AMI_DISPLAY),
]
match_records = []

for ds in SAMPLES:
    for mname, _, _, _, _, label, cat in LEARN_CONFIGS:
        if mname not in matched_segmentations[ds]:
            continue
        remapped_work, ref_segs = matched_segmentations[ds][mname]
        l_work = match.state_lengths(remapped_work)
        l_ref = match.state_lengths(ref_segs)
        
        overlap = match.pair_overlap(ref_segs, remapped_work)
        mdict = match.agreement_metrics(overlap, l_work, l_ref)
        for metric_key, metric_disp in AGREEMENT_METRICS:
            match_records.append({
                "Sample": ds,
                "SampleName": CELL_NAMES[ds],
                "Method": mname,
                "Method_Label": label,
                "Category": cat,
                "Metric": metric_disp,
                "Value": mdict.get(metric_key, np.nan),
            })

df_match = pd.DataFrame(match_records)

# Save reference agreement table
ref_agree_tsv = os.path.join(parameters_dir, "reference_agreement.tsv")
df_match.to_csv(ref_agree_tsv, sep="\t", index=False)
print(f"Reference agreement table saved to: {ref_agree_tsv}")

print("\n--- Reference Agreement Summary (Mean ± SD) ---")
summary_pivot = df_match.pivot_table(index="SampleName", columns="Metric", values="Value", aggfunc=["mean", "std"])
print(summary_pivot.round(3))

# Plotting: Reference Agreement
metric_order = [disp for _, disp in AGREEMENT_METRICS]

fig_ag_comb, ax_ag_comb = plt.subplots(figsize=(7.5, 4.5))
utils.bar_plot(
    df_match,
    x="Metric",
    y="Value",
    order=metric_order,
    palette=summary_plots.AGREEMENT_COLORS,
    ax=ax_ag_comb,
    rotation=0,
    hatch=None,
    ylim=(0, 1.05),
    title="Agreement with Reference across ChromHMM variations",
    xlabel="Metric",
    ylabel="Agreement Score",
    points={"size": 3, "alpha": 0.35, "jitter": 0.15},
)
plot_ag_comb_file = os.path.join(plots_dir, "reference_agreement.pdf")
utils.save_fig(fig_ag_comb, plot_ag_comb_file)

In [ ]:
# Compute Multi-Metric Intra-Sample and Cross-Sample Agreements
# (Composition, Kappa, Jaccard, AMI)

# 1. Intra-sample agreements (Method variations on same sample)
mkeys = [m[0] for m in LEARN_CONFIGS]
intra_agree_records = []

for ds in SAMPLES:
    for i in range(len(mkeys)):
        for j in range(i + 1, len(mkeys)):
            m1, m2 = mkeys[i], mkeys[j]
            s1 = matched_segmentations[ds][m1][0]
            s2 = matched_segmentations[ds][m2][0]
            l1 = match.state_lengths(s1)
            l2 = match.state_lengths(s2)
            ov = match.pair_overlap(s2, s1)
            mdict = match.agreement_metrics(ov, l1, l2)
            for metric_key, metric_disp in AGREEMENT_METRICS:
                intra_agree_records.append({
                    "Sample": ds,
                    "SampleName": CELL_NAMES[ds],
                    "Method1": m1,
                    "Method2": m2,
                    "Method1_Label": METHOD_NAMES[m1],
                    "Method2_Label": METHOD_NAMES[m2],
                    "Metric": metric_disp,
                    "Value": mdict.get(metric_key, np.nan),
                })

df_intra_agree = pd.DataFrame(intra_agree_records)
intra_agree_tsv = os.path.join(parameters_dir, "intra_sample_agreement.tsv")
df_intra_agree.to_csv(intra_agree_tsv, sep="\t", index=False)
print(f"Intra-sample agreement table saved to: {intra_agree_tsv}")

# 2. Cross-sample agreements (Biological sample pairs under same method)
sample_pairs = list(combinations(SAMPLES, 2))
cross_agree_records = []

for mname, _, _, _, _, label, cat in LEARN_CONFIGS:
    for ds1, ds2 in sample_pairs:
        s1 = matched_segmentations[ds1][mname][0]
        s2 = matched_segmentations[ds2][mname][0]
        l1 = match.state_lengths(s1)
        l2 = match.state_lengths(s2)
        ov = match.pair_overlap(s2, s1)
        mdict = match.agreement_metrics(ov, l1, l2)
        for metric_key, metric_disp in AGREEMENT_METRICS:
            cross_agree_records.append({
                "Pair": f"{CELL_NAMES[ds1]} vs {CELL_NAMES[ds2]}",
                "Sample1": ds1,
                "Sample2": ds2,
                "Method": mname,
                "Method_Label": label,
                "Category": cat,
                "Metric": metric_disp,
                "Value": mdict.get(metric_key, np.nan),
            })

df_cross_agree = pd.DataFrame(cross_agree_records)
cross_agree_tsv = os.path.join(parameters_dir, "cross_sample_agreement.tsv")
df_cross_agree.to_csv(cross_agree_tsv, sep="\t", index=False)
print(f"Cross-sample agreement table saved to: {cross_agree_tsv}")

print("\n--- Multi-Metric Summary (Mean ± SD) ---")
print("Intra-sample Agreement:")
print(df_intra_agree.groupby("Metric")["Value"].agg(["mean", "std"]).round(3))

print("\nCross-sample Agreement:")
print(df_cross_agree.groupby("Metric")["Value"].agg(["mean", "std"]).round(3))

# Plotting: Intra-Sample Agreement
fig_intra, ax_intra = plt.subplots(figsize=(7.5, 4.5))
utils.bar_plot(
    df_intra_agree,
    x="Metric",
    y="Value",
    order=metric_order,
    palette=summary_plots.AGREEMENT_COLORS,
    ax=ax_intra,
    rotation=0,
    hatch=None,
    ylim=(0, 1.05),
    title="Intra-Sample Agreement across ChromHMM variations",
    xlabel="Metric",
    ylabel="Agreement Score",
    points={"size": 2.5, "alpha": 0.25, "jitter": 0.15},
)
plot_intra_file = os.path.join(plots_dir, "intra_sample_agreement.pdf")
utils.save_fig(fig_intra, plot_intra_file)

# Plotting: Cross-Sample Agreement
fig_cross, ax_cross = plt.subplots(figsize=(7.5, 4.5))
utils.bar_plot(
    df_cross_agree,
    x="Metric",
    y="Value",
    order=metric_order,
    palette=summary_plots.AGREEMENT_COLORS,
    ax=ax_cross,
    rotation=0,
    hatch=None,
    ylim=(0, 1.05),
    title="Cross-Sample Agreement (Biological Variance)",
    xlabel="Metric",
    ylabel="Agreement Score",
    points={"size": 3, "alpha": 0.35, "jitter": 0.15},
)
plot_cross_file = os.path.join(plots_dir, "cross_sample_agreement.pdf")
utils.save_fig(fig_cross, plot_cross_file)